# VTuber Builder

## ① 환경 설치

In [ ]:
import collections, pathlib, shutil, subprocess, sys, threading

REPO_URL = "https://github.com/jujumelona/Virtual-pipeline.git"
REPO_DIR = pathlib.Path("/content/Virtual-pipeline")
SETUP_LOG = pathlib.Path("/content/vtuber_builder/logs/colab_bootstrap.log")

def run(cmd, timeout):
    SETUP_LOG.parent.mkdir(parents=True, exist_ok=True)
    display_cmd = subprocess.list2cmdline(cmd)
    print(f"[Colab 준비] $ {display_cmd}", flush=True)
    tail = collections.deque(maxlen=80)
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )

    def forward():
        assert proc.stdout is not None
        with SETUP_LOG.open("a", encoding="utf-8") as log:
            log.write(f"\\n[Colab 준비] $ {display_cmd}\\n")
            for line in proc.stdout:
                tail.append(line)
                log.write(line)
                log.flush()
                print(line, end="", flush=True)

    reader = threading.Thread(target=forward, daemon=True)
    reader.start()
    try:
        code = proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired as exc:
        proc.kill()
        proc.wait(timeout=10)
        reader.join(timeout=10)
        raise RuntimeError(f"환경 설치 시간 초과 ({timeout}s), 전체 로그: {SETUP_LOG}") from exc
    reader.join(timeout=10)
    if code:
        raise RuntimeError(
            f"환경 설치 명령 실패 (exit={code}): {display_cmd}\\n"
            + "".join(tail)[-8000:]
            + f"\\n전체 로그: {SETUP_LOG}"
        )

print("①-1 / GitHub main 다운로드 및 동기화", flush=True)
if (REPO_DIR / ".git").is_dir():
    run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], 60)
    run(["git", "-C", str(REPO_DIR), "fetch", "--prune", "origin", "main"], 180)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)], 300)

run(["git", "-C", str(REPO_DIR), "checkout", "-B", "main", "origin/main"], 60)
run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], 60)
print("①-2 / Gradio UI 패키지 설치", flush=True)
run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
     "gradio==6.3.0"], 600)

print("①-3 / 환경 준비", flush=True)
# Install in a CHILD Python process. Pip changes installed packages during
# preparation; the notebook kernel must not retain stale Gradio/NumPy imports.
setup_python = (
    "import os, pathlib, runpy; "
    "os.environ['VTUBER_SETUP_ONLY'] = '1'; "
    "REPO_DIR = pathlib.Path('/content/Virtual-pipeline'); "
    "app = runpy.run_path(str(REPO_DIR / 'tools' / 'colab_app.py'), "
    "run_name='vtuber_prepare'); "
    "head, logs = app['ensure_runtime'](); "
    "print('\\n'.join(logs), flush=True); "
    "print('✅ ① 준비 완료. main=' + head[:12], flush=True)"
)
run([sys.executable, "-u", "-c", setup_python], 9000)
print("✅ 환경 설치 완료", flush=True)


## ② 모델 다운로드·검증

In [ ]:
import collections, pathlib, subprocess, sys, threading

repo = pathlib.Path("/content/Virtual-pipeline")
model_log = pathlib.Path("/content/vtuber_builder/logs/colab_model_setup.log")
model_log.parent.mkdir(parents=True, exist_ok=True)
code = (
    "import os, pathlib, runpy; "
    "os.environ['VTUBER_SETUP_ONLY'] = '1'; "
    "repo = pathlib.Path('/content/Virtual-pipeline'); "
    "app = runpy.run_path(str(repo / 'tools' / 'colab_app.py'), run_name='prepare'); "
    "app['prepare_models']()"
)

print("② 모델 다운로드·초기화 검증", flush=True)
print(f"모델 준비 전체 로그: {model_log}", flush=True)
tail = collections.deque(maxlen=120)
proc = subprocess.Popen(
    [sys.executable, "-u", "-c", code],
    cwd=str(repo),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

def forward_model_output():
    assert proc.stdout is not None
    with model_log.open("a", encoding="utf-8") as out:
        for line in proc.stdout:
            out.write(line)
            out.flush()
            tail.append(line)
            print(line, end="", flush=True)

reader = threading.Thread(target=forward_model_output, daemon=True)
reader.start()
try:
    result_code = proc.wait(timeout=3600)
except subprocess.TimeoutExpired as exc:
    proc.kill()
    proc.wait(timeout=10)
    reader.join(timeout=10)
    raise RuntimeError(f"모델 준비 시간 초과; 전체 로그: {model_log}") from exc
reader.join(timeout=10)
if result_code != 0:
    raise RuntimeError(
        f"모델 준비 실패 (exit={result_code})\n"
        + "".join(tail)[-12000:]
        + f"\n전체 로그: {model_log}"
    )
print("✅ 모델 준비 완료", flush=True)


## ③ 생성 UI

In [ ]:
import pathlib, runpy

repo = pathlib.Path("/content/Virtual-pipeline")
# Keep runpy's namespace out of the Colab cell's displayed result.
_launcher_globals = runpy.run_path(
    str(repo / "tools" / "colab_ui_launcher.py"), run_name="__main__"
)
del _launcher_globals
